# RLHF: From Christiano 2017 to InstructGPT to Today's Recipe

The full code of [RLHF: From Christiano 2017 to InstructGPT to Today's Recipe](https://residual-stream.lvtan71.workers.dev/posts/rlhf-from-comparisons/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/rlhf-from-comparisons) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/rlhf-from-comparisons.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "rlhf-from-comparisons"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `bradley_terry.py`

Bradley-Terry scores for three answers fitted by gradient ascent to 13 pairwise outcomes, and the win probabilities they imply against the observed ones.

In [ ]:
!{sys.executable} bradley_terry.py

The post printed:

```text
rewards: r_A = +1.67, r_B = +0.84, r_C = +0.00
P(A beats B) = sigmoid(1.67 - 0.84) = 0.70   observed 3/4 = 0.75
P(B beats C) = sigmoid(0.84 - 0.00) = 0.70   observed 3/4 = 0.75
P(A beats C) = sigmoid(1.67 - 0.00) = 0.84   observed 4/5 = 0.80
```

## `overoptimization.py`

RLHF solved exactly on 512 three-token responses: a reward model fitted to 300 noisy comparisons from a hidden gold reward, then the KL-regularized optimal policy for eight values of beta, in 100 worlds of each of three kinds. Prints KL, proxy gain, gold gain and sevens per answer.

In [ ]:
!{sys.executable} overoptimization.py

The post printed:

```text

world "random": means over 100 worlds; the starting policy writes 0.37 sevens per answer
  beta  KL from ref  proxy gain  gold gain  7s per answer
     3         0.11        0.69       0.70           0.37
     1         0.88        1.87       1.88           0.37
   0.5         2.39        2.92       2.91           0.37
   0.3         3.76        3.46       3.45           0.36
   0.2         4.67        3.68       3.67           0.35
   0.1         5.64        3.83       3.82           0.34
  0.05         6.10        3.86       3.87           0.33
  0.02         6.37        3.87       3.88           0.32
worlds whose gold gain ends more than 0.5 below its peak: 6%

world "in moderation, +2 for one 7": means over 100 worlds; the starting policy writes 0.37 sevens per answer
  beta  KL from ref  proxy gain  gold gain  7s per answer
     3         0.11        0.68       0.74           0.47
     1         0.91        1.89       1.92           0.71
   0.5         2.42        2.95       2.75           0.95
   0.3         3.78        3.48       3.10           1.07
   0.2         4.66        3.70       3.24           1.11
   0.1         5.62        3.85       3.35           1.12
  0.05         6.10        3.88       3.40           1.12
  0.02         6.36        3.89       3.41           1.11
worlds whose gold gain ends more than 0.5 below its peak: 22%

world "in moderation, +4 for one 7": means over 100 worlds; the starting policy writes 0.37 sevens per answer
  beta  KL from ref  proxy gain  gold gain  7s per answer
     3         0.12        0.71       0.88           0.55
     1         1.06        2.15       1.92           1.02
   0.5         2.83        3.39       1.69           1.55
   0.3         4.20        3.93       1.05           1.85
   0.2         4.94        4.12       0.67           1.98
   0.1         5.66        4.22       0.35           2.07
  0.05         6.03        4.25       0.19           2.10
  0.02         6.29        4.26       0.10           2.11
worlds whose gold gain ends more than 0.5 below its peak: 71%
```